Data Engineering Interview Prep — PySpark Practice
====================================================

Same 17 problems, now in PySpark.
Each problem has:
  - DataFrame creation with sample data
  - Your solution stub (TODO)
  - Expected results as comments
  - Verification code

Setup: pip install pyspark
Run:   python interview_prep_pyspark.py

NOTE: These use the DataFrame API (not RDDs). This is what production
Spark code looks like and what interviewers expect.

In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("interview_prep") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/30 11:16:40 WARN Utils: Your hostname, Yogis-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.69 instead (on interface en0)
26/03/30 11:16:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 11:16:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# ============================================================================
# P1. Group and Aggregate — total spend per user per category
# Expected:
#   u1 | electronics | 80.0
#   u1 | books       | 20.0
#   u2 | electronics | 100.0
#   u2 | books       | 15.0
#   u3 | books       | 5.0
# ============================================================================

p1_data = [
    ("u1", 50.0, "electronics"),
    ("u1", 30.0, "electronics"),
    ("u1", 20.0, "books"),
    ("u2", 100.0, "electronics"),
    ("u2", 15.0, "books"),
    ("u3", 5.0, "books"),
]
p1_df = spark.createDataFrame(p1_data, ["user_id", "amount", "category"])

def p1_group_and_aggregate(df):
    # TODO: return DataFrame with columns: user_id, category, total_amount
    return df.groupBy("user_id", "category").agg(F.sum("amount").alias("amount"),)

result = p1_group_and_aggregate(p1_df)
result.show()

+-------+-----------+------+
|user_id|   category|amount|
+-------+-----------+------+
|     u1|electronics|  80.0|
|     u1|      books|  20.0|
|     u2|electronics| 100.0|
|     u2|      books|  15.0|
|     u3|      books|   5.0|
+-------+-----------+------+



In [ ]:
# ============================================================================
# P2. Sort with Tiebreakers — score desc, then earliest timestamp
# Expected order: Diana, Bob, Charlie, Alice, Eve
# ============================================================================

p2_data = [
    ("Alice",   90, "2024-01-03T10:00:00"),
    ("Bob",     95, "2024-01-02T10:00:00"),
    ("Charlie", 90, "2024-01-01T10:00:00"),
    ("Diana",   95, "2024-01-01T10:00:00"),
    ("Eve",     85, "2024-01-01T10:00:00"),
]
p2_df = spark.createDataFrame(p2_data, ["name", "score", "ts"])

def p2_sort_with_tiebreakers(df):
    # TODO: return DataFrame sorted by score desc, ts asc
    pass

# result = p2_sort_with_tiebreakers(p2_df)
# result.show()

In [ ]:
# ============================================================================
# P3. Deduplication — one record per email, earliest signup, prefer organic
# Expected:
#   a@test.com | 2024-01-01 | organic
#   b@test.com | 2024-01-05 | paid
#   c@test.com | 2024-01-01 | paid
# ============================================================================

p3_data = [
    ("a@test.com", "2024-01-01", "paid"),
    ("a@test.com", "2024-01-01", "organic"),
    ("a@test.com", "2024-01-02", "organic"),
    ("b@test.com", "2024-01-05", "paid"),
    ("c@test.com", "2024-01-03", "organic"),
    ("c@test.com", "2024-01-01", "paid"),
]
p3_df = spark.createDataFrame(p3_data, ["email", "signup_date", "source"])

def p3_deduplicate(df):
    # TODO: return one row per email — earliest signup_date, organic preferred on tie
    # hint: window + row_number with careful ordering
    pass

# result = p3_deduplicate(p3_df)
# result.show()


In [ ]:
# ============================================================================
# P4. Merge Two Sorted Streams
# Expected order: b1, a1, a2, b2, b3, a3
# ============================================================================

p4_a = spark.createDataFrame([
    ("2024-01-01T10:00:00", "a1"),
    ("2024-01-01T12:00:00", "a2"),
    ("2024-01-01T14:00:00", "a3"),
], ["ts", "value"])

p4_b = spark.createDataFrame([
    ("2024-01-01T09:00:00", "b1"),
    ("2024-01-01T12:00:00", "b2"),
    ("2024-01-01T13:00:00", "b3"),
], ["ts", "value"])

def p4_merge_streams(df_a, df_b):
    # TODO: union and sort by ts, with stream_a before stream_b on tie
    # hint: add a source column before union, use it as tiebreaker
    pass

# result = p4_merge_streams(p4_a, p4_b)
# result.show()

In [ ]:
# ============================================================================
# P5. Window Lookback — add prev_action per user
# Expected:
#   u1 | login    | NULL
#   u2 | view     | NULL
#   u1 | click    | login
#   u1 | purchase | click
#   u2 | click    | view
# ============================================================================

p5_data = [
    ("u1", "login",    "2024-01-01T10:00:00"),
    ("u2", "view",     "2024-01-01T10:01:00"),
    ("u1", "click",    "2024-01-01T10:02:00"),
    ("u1", "purchase", "2024-01-01T10:05:00"),
    ("u2", "click",    "2024-01-01T10:06:00"),
]
p5_df = spark.createDataFrame(p5_data, ["user_id", "action", "ts"])

def p5_add_prev_action(df):
    # TODO: add prev_action column using lag window function
    pass

# result = p5_add_prev_action(p5_df)
# result.show()

In [ ]:
# ============================================================================
# P6. Detect Status Changes
# Expected for u1: active(10:00), inactive(12:00), active(13:00)
# Expected for u2: active(10:00) only
# ============================================================================

p6_data = [
    ("u1", "active",   "2024-01-01T10:00:00"),
    ("u1", "active",   "2024-01-01T11:00:00"),
    ("u1", "inactive", "2024-01-01T12:00:00"),
    ("u1", "active",   "2024-01-01T13:00:00"),
    ("u2", "active",   "2024-01-01T10:00:00"),
    ("u2", "active",   "2024-01-01T11:00:00"),
    # unsorted u3
    ("u3", "inactive", "2024-01-01T12:00:00"),
    ("u3", "active",   "2024-01-01T10:00:00"),
    ("u3", "active",   "2024-01-01T11:00:00"),
]
p6_df = spark.createDataFrame(p6_data, ["user_id", "status", "ts"])

def p6_detect_status_changes(df):
    # TODO: return only rows where status changed from previous (or first event)
    # hint: lag(status) over (partition by user_id order by ts), filter where different or null
    pass

# result = p6_detect_status_changes(p6_df)
# result.show()

In [ ]:
# ============================================================================
# P7. Session Builder — new session if >30 min gap
# Expected:
#   u1 | session 1 | 10:00 - 10:25 | 3 events
#   u1 | session 2 | 11:00 - 11:00 | 1 event
#   u2 | session 1 | 10:00 - 10:00 | 1 event
# ============================================================================

p7_data = [
    ("u1", "click", "2024-01-01T10:00:00"),
    ("u1", "view",  "2024-01-01T10:15:00"),
    ("u1", "click", "2024-01-01T10:25:00"),
    ("u1", "view",  "2024-01-01T11:00:00"),
    ("u2", "click", "2024-01-01T10:00:00"),
]
p7_df = spark.createDataFrame(p7_data, ["user_id", "event", "ts"])

def p7_build_sessions(df, gap_minutes=30):
    # TODO: return DataFrame with user_id, session_id, start, end, event_count
    # hint:
    #   1. lag(ts) to get prev_ts per user
    #   2. flag new_session = 1 where gap > 30 min or first event
    #   3. sum(new_session) as running window → session_id
    #   4. group by user_id, session_id → min(ts), max(ts), count(*)
    pass

# result = p7_build_sessions(p7_df)
# result.show()

In [ ]:
# ============================================================================
# P8. First and Last Per Group
# Expected:
#   d1 | first_reading=22.5 | last_reading=24.0 | count=4
#   d2 | first_reading=18.0 | last_reading=18.0 | count=1
# ============================================================================

p8_data = [
    ("d1", 22.5, "2024-01-01T10:00:00"),
    ("d1", 23.1, "2024-01-01T12:00:00"),
    ("d1", None, "2024-01-01T11:00:00"),
    ("d2", 18.0, "2024-01-01T09:00:00"),
    ("d1", 24.0, "2024-01-01T14:00:00"),
]
p8_df = spark.createDataFrame(p8_data, ["device_id", "reading", "ts"])

def p8_first_last_per_device(df):
    # TODO: return device_id, first_reading, last_reading, count
    # hint: first_value/last_value window functions, or join with min/max ts subqueries
    pass

# result = p8_first_last_per_device(p8_df)
# result.show()

In [ ]:
# ============================================================================
# P9. Running State Tracker — threshold crossings
# Crossing = value goes from <=100 to >100
# Expected:
#   m1 | temp     | 10:05 | 102 (was 95)
#   m1 | temp     | 10:15 | 105 (was 98)
#   m1 | pressure | 10:10 | 101 (was 90)
# ============================================================================

p9_data = [
    ("m1", "temp",     95.0,  "2024-01-01T10:00:00"),
    ("m1", "temp",     102.0, "2024-01-01T10:05:00"),
    ("m1", "temp",     98.0,  "2024-01-01T10:10:00"),
    ("m1", "temp",     105.0, "2024-01-01T10:15:00"),
    ("m1", "pressure", 110.0, "2024-01-01T10:00:00"),
    ("m1", "pressure", 90.0,  "2024-01-01T10:05:00"),
    ("m1", "pressure", 101.0, "2024-01-01T10:10:00"),
]
p9_df = spark.createDataFrame(p9_data, ["machine_id", "metric", "value", "ts"])

def p9_detect_threshold_crossings(df, threshold=100):
    # TODO: return rows where value crossed above threshold
    # hint: lag(value) partitioned by machine_id, metric; filter prev<=threshold and curr>threshold
    pass

# result = p9_detect_threshold_crossings(p9_df)
# result.show()

In [ ]:
# ============================================================================
# P10. Time Gap Detector — sensor gaps > 5 minutes
# Expected:
#   s1 | gap_start=10:03 | gap_end=10:12 | gap_sec=540
# ============================================================================

p10_data = [
    ("s1", "2024-01-01T10:00:00"),
    ("s1", "2024-01-01T10:03:00"),
    ("s1", "2024-01-01T10:12:00"),
    ("s1", "2024-01-01T10:14:00"),
    ("s2", "2024-01-01T10:00:00"),
    ("s2", "2024-01-01T10:01:00"),
    ("s3", "2024-01-01T10:00:00"),
]
p10_df = spark.createDataFrame(p10_data, ["sensor_id", "ts"])

def p10_detect_gaps(df, max_gap_seconds=300):
    # TODO: return sensor_id, gap_start, gap_end, gap_duration_sec
    # hint: lag(ts), compute diff in seconds, filter > max_gap_seconds
    pass

# result = p10_detect_gaps(p10_df)
# result.show()

In [ ]:
# ============================================================================
# P11. Event Funnel
# Steps: page_view → add_to_cart → checkout → purchase
# Step counts only if it happened AFTER previous step
# Expected:
#   u1 | purchase    | completed=true
#   u2 | add_to_cart | completed=false
#   u3 | page_view   | completed=false
#   u4 | add_to_cart | completed=false
#   u5 | add_to_cart | completed=false
# ============================================================================

p11_data = [
    ("u1", "page_view",   "2024-01-01T10:00:00"),
    ("u1", "add_to_cart", "2024-01-01T10:05:00"),
    ("u1", "checkout",    "2024-01-01T10:10:00"),
    ("u1", "purchase",    "2024-01-01T10:15:00"),
    ("u2", "page_view",   "2024-01-01T10:00:00"),
    ("u2", "add_to_cart", "2024-01-01T10:05:00"),
    ("u3", "page_view",   "2024-01-01T10:00:00"),
    ("u3", "checkout",    "2024-01-01T10:05:00"),
    ("u4", "page_view",   "2024-01-01T10:00:00"),
    ("u4", "add_to_cart", "2024-01-01T10:05:00"),
    ("u4", "purchase",    "2024-01-01T10:06:00"),
    ("u4", "checkout",    "2024-01-01T10:10:00"),
    ("u5", "page_view",   "2024-01-01T10:00:00"),
    ("u5", "page_view",   "2024-01-01T10:01:00"),
    ("u5", "add_to_cart", "2024-01-01T10:05:00"),
]
p11_df = spark.createDataFrame(p11_data, ["user_id", "event_type", "ts"])

def p11_compute_funnel(df):
    # TODO: return user_id, furthest_step, completed_funnel
    # hint: pivot to get min ts per user per step, then chain comparisons
    pass

# result = p11_compute_funnel(p11_df)
# result.show()

In [ ]:
# ============================================================================
# P12. SLA Breach Detector — opened→resolved > 4 hours = breach
# Expected:
#   t1 | 2.0 hrs  | breached=false
#   t2 | 5.0 hrs  | breached=true
#   t4 | 1.0 hrs  | breached=false
# ============================================================================

p12_data = [
    ("t1", "opened",   "2024-01-01T10:00:00"),
    ("t1", "assigned", "2024-01-01T10:30:00"),
    ("t1", "resolved", "2024-01-01T12:00:00"),
    ("t2", "opened",   "2024-01-01T08:00:00"),
    ("t2", "resolved", "2024-01-01T13:00:00"),
    ("t3", "opened",   "2024-01-01T09:00:00"),
    ("t3", "assigned", "2024-01-01T09:30:00"),
    ("t4", "opened",   "2024-01-01T10:00:00"),
    ("t4", "opened",   "2024-01-01T10:01:00"),
    ("t4", "resolved", "2024-01-01T11:00:00"),
    ("t5", "assigned", "2024-01-01T10:00:00"),
    ("t5", "resolved", "2024-01-01T11:00:00"),
]
p12_df = spark.createDataFrame(p12_data, ["ticket_id", "status", "ts"])

def p12_detect_sla_breaches(df, sla_hours=4):
    # TODO: return ticket_id, open_time, resolve_time, duration_hrs, breached
    # hint: pivot min ts per status per ticket, compute duration
    pass

# result = p12_detect_sla_breaches(p12_df)
# result.show()

In [ ]:
# ============================================================================
# P13. Daily Active Users with Day-1 Retention
# Expected:
#   2024-01-01 | dau=3 | d1_retention=66.67%
#   2024-01-02 | dau=3 | d1_retention=33.33%
#   2024-01-03 | dau=1 | d1_retention=NULL
# ============================================================================

p13_data = [
    ("u1", "click", "2024-01-01"),
    ("u2", "click", "2024-01-01"),
    ("u3", "click", "2024-01-01"),
    ("u1", "click", "2024-01-01"),
    ("u1", "view",  "2024-01-02"),
    ("u2", "click", "2024-01-02"),
    ("u4", "click", "2024-01-02"),
    ("u4", "view",  "2024-01-03"),
]
p13_df = spark.createDataFrame(p13_data, ["user_id", "event", "event_date"])

def p13_compute_dau_retention(df):
    # TODO: return event_date, dau, d1_retention_pct
    # hint: distinct users per day, self-join day to day+1, count overlap
    pass

# result = p13_compute_dau_retention(p13_df)
# result.show()

In [ ]:
# ============================================================================
# P14. Change Data Capture Replay
# Expected final state:
#   users | pk=1 | Alice | alice@new.com
#   users | pk=3 | Charlie | c@test.com
#   orders | pk=100 | user_id=1 | total=50
# ============================================================================

p14_data = [
    ("users",  "1",   "insert", "Alice",   "a@test.com",   None, None, "2024-01-01T10:00:00"),
    ("users",  "2",   "insert", "Bob",     "b@test.com",   None, None, "2024-01-01T10:01:00"),
    ("users",  "1",   "update", "Alice",   "alice@new.com",None, None, "2024-01-01T10:05:00"),
    ("users",  "2",   "delete", None,      None,           None, None, "2024-01-01T10:10:00"),
    ("orders", "100", "insert", None,      None,           "1",  50,   "2024-01-01T10:02:00"),
    ("users",  "3",   "update", "Charlie", "c@test.com",   None, None, "2024-01-01T10:05:00"),
    ("users",  "3",   "insert", "Charles", "c@old.com",    None, None, "2024-01-01T10:00:00"),
    ("users",  "4",   "insert", "Diana",   None,           None, None, "2024-01-01T10:00:00"),
    ("users",  "4",   "delete", None,      None,           None, None, "2024-01-01T10:01:00"),
    ("users",  "4",   "delete", None,      None,           None, None, "2024-01-01T10:02:00"),
]
p14_df = spark.createDataFrame(p14_data,
    ["tbl", "pk", "op", "data_name", "data_email", "data_user_id", "data_total", "ts"])

def p14_replay_cdc(df):
    # TODO: return tbl, pk, and latest non-deleted row data
    # hint: row_number partitioned by tbl+pk ordered by ts desc, take rn=1, filter out deletes
    pass

# result = p14_replay_cdc(p14_df)
# result.show()

In [ ]:
# ============================================================================
# P15. Slowly Changing Dimension Type 2
# Expected for e1:
#   Engineering | 2023-01-01 | 2023-06-01 | false
#   Product     | 2023-06-01 | 2024-01-01 | false
#   Engineering | 2024-01-01 | NULL       | true
# ============================================================================

p15_data = [
    ("e1", "Engineering", "2023-01-01"),
    ("e1", "Product",     "2023-06-01"),
    ("e1", "Engineering", "2024-01-01"),
    ("e2", "Sales",       "2023-03-01"),
    ("e3", "HR",          "2024-01-01"),
    ("e3", "Finance",     "2023-01-01"),
]
p15_df = spark.createDataFrame(p15_data, ["employee_id", "department", "effective_date"])

def p15_build_scd2(df):
    # TODO: return employee_id, department, start_date, end_date, is_current
    # hint: lead(effective_date) over partition by employee_id order by effective_date
    pass

# result = p15_build_scd2(p15_df)
# result.orderBy("employee_id", "start_date").show()

In [ ]:
# ============================================================================
# P16. Rate Limiter Check — >100 requests in 60-second window
# Expected: key1 + /api/data has breaches, others don't
# ============================================================================

from datetime import datetime, timedelta
import pandas as pd

# Generate data using pandas then convert
rows_16 = []
base = datetime(2024, 1, 1, 10, 0, 0)
for i in range(101):
    rows_16.append(("key1", "/api/data", (base + timedelta(seconds=i*0.5)).isoformat()))
for i in range(50):
    rows_16.append(("key2", "/api/data", (base + timedelta(seconds=i*2)).isoformat()))
for i in range(10):
    rows_16.append(("key1", "/api/health", (base + timedelta(seconds=i)).isoformat()))

p16_df = spark.createDataFrame(rows_16, ["api_key", "endpoint", "ts"])

def p16_detect_rate_breaches(df, max_requests=100, window_seconds=60):
    # TODO: return api_key, endpoint, event_ts, request_count_in_window for breaches
    # hint: self-join where ts_b between ts_a - 60s and ts_a, group by ts_a, count > 100
    # OR: use window range frame (trickier in Spark — needs casting ts to long)
    pass

# result = p16_detect_rate_breaches(p16_df)
# result.show()

In [ ]:
# ============================================================================
# P17. Log-Level Anomaly Spotter
# Flag hours where error rate > 2x service's overall average
# Expected: svc-a hour 14 is anomaly
# ============================================================================

rows_17 = []
base = datetime(2024, 1, 1, 0, 0, 0)
for hour in range(24):
    for i in range(90):
        rows_17.append(("svc-a", "INFO", (base + timedelta(hours=hour, minutes=i % 60)).isoformat()))
    error_count = 10 if hour != 14 else 50
    for i in range(error_count):
        rows_17.append(("svc-a", "ERROR", (base + timedelta(hours=hour, minutes=30 + i % 30)).isoformat()))
for hour in range(24):
    for i in range(100):
        rows_17.append(("svc-b", "INFO", (base + timedelta(hours=hour, minutes=i % 60)).isoformat()))

p17_df = spark.createDataFrame(rows_17, ["service", "log_level", "ts"])

def p17_detect_log_anomalies(df):
    # TODO: return service, hour, error_rate, avg_error_rate, is_anomaly
    # hint:
    #   1. extract hour, compute hourly total and error count per service
    #   2. error_rate = errors / total per hour
    #   3. avg_error_rate = avg of error_rate per service (across all hours)
    #   4. flag where hourly > 2 * avg
    pass

# result = p17_detect_log_anomalies(p17_df)
# result.filter(F.col("is_anomaly")).show()

In [ ]:
 spark.stop()